In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

df = pd.read_csv("../cleaned_data/cleaned_city_day.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df[["City","Date","AQI"]].dropna()

cities = df["City"].unique()
results = []

for city in cities:
    city_df = df[df["City"] == city][["Date","AQI"]]
    city_df = city_df.groupby("Date").mean()
    city_df = city_df.asfreq("D")
    city_df["AQI"] = city_df["AQI"].interpolate()

    if len(city_df) < 200:
        continue

    train = city_df[:-30]
    test = city_df[-30:]

    try:
        model = ARIMA(train["AQI"], order=(5,1,0))
        model_fit = model.fit()
        forecast = model_fit.forecast(steps=30)

        rmse = np.sqrt(mean_squared_error(test["AQI"], forecast))
        mae = mean_absolute_error(test["AQI"], forecast)

        results.append([city, rmse, mae])

    except:
        continue

eval_df = pd.DataFrame(results, columns=["City","ARIMA_RMSE","ARIMA_MAE"])
eval_df.to_csv("../forecasts/model_evaluation_all_cities.csv", index=False)

print("Evaluation saved!")

Evaluation saved!
